# Определение токсичных комментариев

**Цель:** предупреждать участников чата о токсичных и оскорбительных комментариях, фильтровать комментарии с негативным содержимым


## Анализ датасета Toxic Russian Comments

Ссылка на датасет:
https://www.kaggle.com/datasets/alexandersemiletov/toxic-russian-comments/data

In [1]:
%pip install fg-data-profiling

In [8]:
import pandas as pd
import numpy as np
import data_profiling
from data_profiling import ProfileReport
 # Для автоматического анализа данных

In [4]:
data_list = []
with open("/content/dataset.txt") as file:
    for line in file:
        labels = line.split()[0]
        text = line[len(labels)+1:].strip()
        labels = labels.split(",")
        mask = [1 if "__label__NORMAL" in labels else 0,
                1 if "__label__INSULT" in labels else 0,
                1 if "__label__THREAT" in labels else 0,
                1 if "__label__OBSCENITY" in labels else 0]
        data_list.append((text, *mask))

df = pd.DataFrame(data_list, columns=["text", "normal", "insult", "threat", "obscenity"])

print("Размер датасета:", df.shape)
df.head()

Размер датасета: (248290, 5)


,text,normal,insult,threat,obscenity
0,скотина! что сказать,0,1,0,0
1,я сегодня проезжала по рабочей и между домами ...,1,0,0,0
2,очередной лохотрон. зачем придумывать очередно...,1,0,0,0
3,"ретро дежавю ... сложно понять чужое сердце , ...",1,0,0,0
4,а когда мы статус агрогородка получили?,1,0,0,0


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 248290 entries, 0 to 248289
Data columns (total 5 columns):
 #   Column     Non-Null Count   Dtype 
---  ------     --------------   ----- 
 0   text       248290 non-null  object
 1   normal     248290 non-null  int64 
 2   insult     248290 non-null  int64 
 3   threat     248290 non-null  int64 
 4   obscenity  248290 non-null  int64 
dtypes: int64(4), object(1)
memory usage: 9.5+ MB


In [9]:
# СОЗДАНИЕ ДЕТАЛЬНОГО ОТЧЕТА ПО ДАННЫМ С ПОМОЩЬЮ YDATA_PROFILING
profile = ProfileReport(df, title="Profiling Report")
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 5/5 [00:12<00:00,  2.55s/it]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Датасет состоит из 248290 комментариев, все комментарии отнесены к одному из классов - normal, unsult, threat, obscenity. Разметка датасета качественная, пропусков нет.

По данным автоматического отчета можем сказать, что в датасете есть три проблемы:
- найдены дубликаты данных - необходимо удалить дубликаты строк перед обучением модели
- есть корреляция между классами - необходимо проанализировать и устранить причины, если возможно
- есть дисбаланс классов - необходимо сбалансировать классы перед обучением модели.



## Анализ датасета russian-inappropriate-messages

Ссылка на датасет:
https://www.kaggle.com/datasets/nigula/russianinappropriatemessages

In [10]:
df2 = pd.read_csv('/content/Inappapropriate_messages.csv')

print("Размер датасета:", df2.shape)
df2.head()

Размер датасета: (124597, 2)


,text,inappropriate
0,"Бедный Ниссон, его бесконечных детей то похища...",0.9526
1,Все кто лишился девстенности после NUMBER - ом...,0.9972
2,Не знаю сам почему её туда отправили. Не торга...,0.0217
3,Проституция легальнаНелегальное предпринимател...,0.9986
4,"Я бы повстречался с порноактрисой, только я не...",0.9998


In [12]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 124597 entries, 0 to 124596
Data columns (total 2 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   text           124597 non-null  object 
 1   inappropriate  124597 non-null  float64
dtypes: float64(1), object(1)
memory usage: 1.9+ MB


In [13]:
profile = ProfileReport(df2, title="Profiling Report")
profile

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 2/2 [00:07<00:00,  3.80s/it]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Датасет состоит из 124597 комментариев, каждый комментарий размечен по степени токсичности от 0 до 1. Разметка датасета качественная, пропусков нет, дубликатов нет.

Автоматический отчет не нашел проблем в данных. Однако при обучении модели стоит определить точку отсечения для попадания комментария в категорию "токсичный" и разметить датасет соответственно.